# Prepare the training data -- once, on CPU

Downloads **Middlebury** (every release with public ground truth, both views' disparity),
**KITTI 2012 + 2015** (the 394 labelled training pairs) and the official **Scene Flow** --
FlyingThings3D (22,390 training + 4,370 test frames), Monkaa and Driving -- from their own
servers, checks every label by warping, and writes the result to `/kaggle/working/stereo-data`,
about 15 GB. Scene Flow's training frames are halved to 480x270; FlyingThings3D's TEST stays at
960x540, so the paper's Table IV is scored on its own images. Scene Flow is streamed: its
disparity alone is 133 GB compressed, more than the session's disk.

It also **indexes your FlyingThings3D container**: listing its 189,000 arrays takes about ten
minutes, which every training session would otherwise spend with the GPU idle. Here it is done
once, on CPU, and shipped with the data.

Make that output a dataset once, and the training notebook attaches it with **no download and no
indexing in the GPU session**.

**Before running** (right-hand panel): Accelerator **None**, Internet **On**.
Then **Save Version -> Save & Run All**: about 3-4 hours with Scene Flow, nearly all of it
streaming FlyingThings3D's disparity; about 15 minutes with `SCENEFLOW = ()`.

In [ ]:
import os, subprocess, sys

GITHUB_USER, GITHUB_REPO, GITHUB_BRANCH = "hnquang-cs", "stereo-depth", "main"
SCENEFLOW = ("flyingthings3d", "monkaa", "driving")    # () leaves Scene Flow out

# The code goes to /tmp, not /kaggle/working: everything in /kaggle/working becomes
# the notebook's output, and so would end up inside the dataset.
ROOT = f"/tmp/{GITHUB_REPO}"
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--depth", "1", "-b", GITHUB_BRANCH,
                    f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git", ROOT], check=True)
sys.path.insert(0, ROOT)

from stereo.data.prepare import prepare_all

prepare_all("/kaggle/working/stereo-data", cache="/tmp/stereo-prepare-cache",
            index_hdf5_under="/kaggle/input", sceneflow=SCENEFLOW)

## Make it a dataset (once)

1. When the run has finished, open this notebook's **Output** tab: `stereo-data/` is there.
2. Choose **New Dataset** from that output, keep it **Private**, and name it, for example,
   `stereo-training-data`.
3. In the training notebook, **Add Input -> Your Datasets ->** that dataset. Nothing else to edit:
   it is found by its `stereo_data_manifest.json`, wherever Kaggle mounts it.

Keep it private: KITTI is licensed CC BY-NC-SA 3.0, and Middlebury asks to be cited.